# アルゴリズム論Ⅱ 

## 第4回：ガウスの消去法

前回の「第3回：ニュートン法」では、関数の接線の傾きを利用して非線形方程式 $f(x) = 0$ の解を超高速に探索する手法を学びました。  

実社会の**データサイエンス（多変量回帰分析・機械学習）**、電気電子工学、構造力学、物理エンジン等では、未知数が数千〜数百万個におよぶ多変数の問題が日常的に扱われます。多次元ニュートン法 $J(\mathbf{x}_k) \Delta \mathbf{x} = -\mathbf{f}(\mathbf{x}_k)$ をはじめ、これらすべての専門分野の根幹には **連立一次方程式 $A \mathbf{x} = \mathbf{b}$ を解く** という決定的なステップが存在します。

連立一次方程式の最も基本的な直接解法といえば、線形代数で学ぶ **「ガウスの消去法（Gaussian Elimination ）」** です。  
しかし、紙と鉛筆の計算と異なり、有限桁の浮動小数点数（倍精度64bit）で動作するコンピュータ上で素朴（ナイーブ）に消去法を実行すると、**ゼロでの除算（割り算）によるクラッシュ**や、**微小な数値での割り算による桁落ち・情報落ちで解が完全に壊れる**という深刻な問題に直面することがあることには、注意が必要です。

---
## 1. 連立一次方程式と拡大係数行列の基礎

$n$ 個の未知数 $x_0, x_1, \dots, x_{n-1}$ を持つ連立一次方程式は、一般に次のように記述されます：

$$
\begin{cases}
a_{00} x_0 + a_{01} x_1 + \cdots + a_{0, n-1} x_{n-1} = b_0 \\
a_{10} x_0 + a_{11} x_1 + \cdots + a_{1, n-1} x_{n-1} = b_1 \\
\quad \vdots \\
a_{n-1, 0} x_0 + a_{n-1, 1} x_1 + \cdots + a_{n-1, n-1} x_{n-1} = b_{n-1}
\end{cases}
$$

係数行列 $A \in \mathbb{R}^{n \times n}$、未知数ベクトル $\mathbf{x} \in \mathbb{R}^n$、右辺ベクトル $\mathbf{b} \in \mathbb{R}^n$ を用いると、行列表現で簡潔に次のように書けます：

$$ A \mathbf{x} = \mathbf{b} $$

### 拡大係数行列（Augmented Matrix）
消去法を行う際は、係数行列 $A$ の右側に右辺ベクトル $\mathbf{b}$ を並べた **拡大係数行列 $[A \mid \mathbf{b}] \in \mathbb{R}^{n \times (n+1)}$** を変形していきます：

$$
[A \mid \mathbf{b}] = \begin{bmatrix}
a_{00} & a_{01} & \cdots & a_{0, n-1} & \bigm| & b_0 \\
a_{10} & a_{11} & \cdots & a_{1, n-1} & \bigm| & b_1 \\
\vdots & \vdots & \ddots & \vdots & \bigm| & \vdots \\
a_{n-1, 0} & a_{n-1, 1} & \cdots & a_{n-1, n-1} & \bigm| & b_{n-1}
\end{bmatrix}
$$

### 行基本変形（Elementary Row Operations）
連立方程式の解集合を変えずに拡大係数行列を変形する操作（行基本変形）は次の3つです：
1. **行の定数倍**: ある行を $c$ 倍（$c \neq 0$）する
2. **行の定数倍の加減算**: ある行から別の行の定数倍を引く（$R_i \leftarrow R_i - m R_k$）
3. **行の入れ替え**: 2つの行を交換する（$R_i \longleftrightarrow R_k$）

### ガウスの消去法の2つのフェーズ
1. **前進消去（Forward Elimination）**:  
   行基本変形を繰り返し、対角成分より下のすべての要素（下三角部分）を $0$ にして、上三角行列（Upper Triangular Matrix）へと変換する。
2. **後退代入（Back Substitution）**:  
   最下行の式から未知数 $x_{n-1}$ の値を確定させ、その値を上の行へと次々に代入して、すべての未知数 $x_{n-2}, \dots, x_0$ を決定する。

---
## 2. ナイーブ（素朴）なガウスの消去法（ピボット選択なし）

まずは最も素朴なアルゴリズムを定式化し、実装します。

### 前進消去の数理アルゴリズム
ステップ $k$（$k = 0, 1, \dots, n-2$）において、対角成分 $a_{kk}^{(k)}$ を **ピボット（主軸）** とし、$k$ 列目の $i$ 行目（$i = k+1, \dots, n-1$）の要素を $0$ に消去します。

各行 $i$ に対する消去乗数（Multiplier）:
$$ m_{ik} = \frac{a_{ik}^{(k)}}{a_{kk}^{(k)}} $$

各要素の更新式（$j = k, \dots, n-1$）：
$$ a_{ij}^{(k+1)} = a_{ij}^{(k)} - m_{ik} a_{kj}^{(k)} $$
$$ b_i^{(k+1)} = b_i^{(k)} - m_{ik} b_k^{(k)} $$

### 後退代入の数理アルゴリズム
すべての前進消去が完了すると、連立方程式は上三角系になります：  
最下行：
$$ x_{n-1} = \frac{b_{n-1}}{a_{n-1, n-1}} $$
第 $i$ 行（$i = n-2, \dots, 0$）：
$$ x_i = \frac{1}{a_{ii}} \left( b_i - \sum_{j=i+1}^{n-1} a_{ij} x_j \right) $$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import math
import japanize_matplotlib

def gaussian_elimination_naive(A, b, verbose=True):
    """
    ナイーブなガウスの消去法（ピボット選択なし）
    A: n x n の係数行列
    b: n 次元の右辺ベクトル
    verbose: 途中経過を表示するかどうか
    """
    A = np.array(A, dtype=float)
    b = np.array(b, dtype=float)
    n = len(b)
    
    if A.shape != (n, n):
        raise ValueError(f"係数行列 A の形状 {A.shape} と ベクトル b の次元 {n} が一致しません。")
    
    # 拡大係数行列 [A | b] の作成
    Ab = np.hstack([A, b.reshape(-1, 1)])
    
    fmt = lambda x: f"{x:12.4e}" if ((abs(x) < 1e-3 and x != 0) or abs(x) >= 1e4) else f"{x:12.4f}"
    
    if verbose:
        print("【初期の拡大係数行列 [A | b]】")
        print(np.array2string(Ab, formatter={'float_kind': fmt}))
        print("-" * 65)

    # 1. 前進消去 (Forward Elimination)
    for k in range(n - 1):
        pivot = Ab[k, k]
        if pivot == 0.0:
            raise ZeroDivisionError(f"ピボット a_{{{k},{k}}} が 0 のため消去を継続できません。")
        
        for i in range(k + 1, n):
            m = Ab[i, k] / Ab[k, k]
            Ab[i, k:] -= m * Ab[k, k:]
            if verbose:
                m_str = f"{m:.4e}" if (abs(m) >= 1e4 or (abs(m) < 1e-3 and m != 0)) else f"{m:.4f}"
                print(f"Step {k+1}: 第 {i} 行 <- 第 {i} 行 - ({m_str}) * 第 {k} 行")
        
        if verbose:
            print(f"--- Step {k+1} 終了後の拡大係数行列 ---")
            print(np.array2string(Ab, formatter={'float_kind': fmt}))
            print("-" * 65)

    # 2. 後退代入 (Back Substitution)
    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        if Ab[i, i] == 0.0:
            raise ZeroDivisionError(f"対角成分 a_{{{i},{i}}} が 0 のため解を一意に決定できません。")
        x[i] = (Ab[i, -1] - np.dot(Ab[i, i+1:n], x[i+1:n])) / Ab[i, i]

    return x

# 具体例の連立一次方程式
#  x0 + 2*x1 +   x2 =  3
# 2*x0 -   x1 + 3*x2 = 14
# 3*x0 +   x1 -   x2 =  2
A_example = [
    [1.0,  2.0,  1.0],
    [2.0, -1.0,  3.0],
    [3.0,  1.0, -1.0]
]
b_example = [3.0, 14.0, 2.0]

print("=== ナイーブなガウスの消去法の実行 ===")
x_sol = gaussian_elimination_naive(A_example, b_example, verbose=True)

print(f"求まった解 x = {x_sol}")
print(f"期待される真の解: [ 2. -1.  3.]")

---
## 3. 落とし穴 1：ゼロピボットによるクラッシュ

ナイーブな消去法では、前進消去の乗数は $m_{ik} = \frac{a_{ik}}{a_{kk}}$ です。  
もし計算途中で **$a_{kk} = 0$** になると、ゼロ除算が発生しプログラムは即座に `ZeroDivisionError` でクラッシュします。

### 恐ろしい事実：解が存在するのに解けない！
「$a_{kk} = 0$ になるのは行列が特異（解が存在しない、または無数にある）だからでは？」と思うかもしれませんが、それは完全な誤解です。  
行列式 $\det(A) \neq 0$ で正則であり、綺麗な唯一の解が存在する場合でも、$a_{kk} = 0$ は容易に発生します。

$$
\begin{bmatrix} 0 & 2 \\ 3 & 4 \end{bmatrix}
\begin{bmatrix} x_0 \\ x_1 \end{bmatrix}
=
\begin{bmatrix} 4 \\ 11 \end{bmatrix}
$$

行列式は $\det(A) = 0 \cdot 4 - 2 \cdot 3 = -6 \neq 0$ です。真の解は明らかに $x_0 = 1$、$x_1 = 2$ です。  
しかし、ナイーブ消去法では最初のピボット $a_{00} = 0$ のため、**1ステップ目すら踏み出せずにクラッシュ**します。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import math
import japanize_matplotlib

def gaussian_elimination_naive(A, b, verbose=True):
    A = np.array(A, dtype=float)
    b = np.array(b, dtype=float)
    n = len(b)
    Ab = np.hstack([A, b.reshape(-1, 1)])
    for k in range(n - 1):
        if Ab[k, k] == 0.0:
            raise ZeroDivisionError(f"ピボット a_{{{k},{k}}} が 0 のため消去を継続できません。")
        for i in range(k + 1, n):
            m = Ab[i, k] / Ab[k, k]
            Ab[i, k:] -= m * Ab[k, k:]
    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        if Ab[i, i] == 0.0:
            raise ZeroDivisionError(f"対角成分 a_{{{i},{i}}} が 0 のため解を一意に決定できません。")
        x[i] = (Ab[i, -1] - np.dot(Ab[i, i+1:n], x[i+1:n])) / Ab[i, i]
    return x

# 行列式 det(A) = -6 != 0 だが a_00 = 0 の方程式系
A_zero_pivot = [
    [0.0, 2.0],
    [3.0, 4.0]
]
b_zero_pivot = [4.0, 11.0]

print("=== ゼロピボットの実験 ===")
print("係数行列 A:")
print(np.array(A_zero_pivot))
print(f"行列式 det(A) = {np.linalg.det(A_zero_pivot):.2f} (正則・唯一の解が存在する)")
print("-" * 65)

try:
    x_fail = gaussian_elimination_naive(A_zero_pivot, b_zero_pivot, verbose=True)
    print(f"求まった解: {x_fail}")
except ZeroDivisionError as e:
    print(f"💥 エラーの内容: {e}")
    print("❌ ピボット a_00 = 0 のため、第1行と第2行の消去計算でゼロ除算が発生しました！")
    print("💡 行 0 と 行 1 を入れ替えるだけで簡単に解けますが、それをしないと即座にクラッシュします。")


---
## 4. 落とし穴 2：微小ピボットによる情報落ち・桁落ち（誤差爆発）

ピボットが $0$ ではなくとも、**「他の成分に比べて非常に小さい値（$a_{kk} \approx 0$）」** である場合、さらに深刻な事態が発生します。

### 例題
次の $2 \times 2$ 連立一次方程式を考えます（$\epsilon > 0$ は極小値）：
$$
\begin{bmatrix} \epsilon & 1 \\ 1 & 1 \end{bmatrix}
\begin{bmatrix} x_0 \\ x_1 \end{bmatrix}
=
\begin{bmatrix} 1 \\ 2 \end{bmatrix}
$$

真の解は $x_0 = \frac{1}{1 - \epsilon} \approx 1.0$、$x_1 = \frac{1 - 2\epsilon}{1 - \epsilon} \approx 1.0$ です。  
しかし、ナイーブ消去法では消去乗数が $m = \frac{1}{\epsilon} = 10^{16}$ となり、天文学的に巨大な数になります。  
倍精度浮動小数点の有効数字は約16桁しかないため、元の行にあった数値 $1$ や $2$ に巨大な数を足し引きすると：
$$ 1 - 10^{16} \approx -10^{16} \quad (1 \text{ が情報落ちして消失}) $$
$$ 2 - 10^{16} \approx -10^{16} \quad (2 \text{ が情報落ちして消失}) $$
後退代入を行うと $x_1 = 1.0$ は求まりますが、$x_0 = \frac{1 - x_1}{\epsilon}$ を計算する際、丸め誤差により $x_0 = 2.22\dots$ や $0.0$ となり、**真値 $1.0$ とは全く異なるデタラメな値**が算出されます！

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import math
import japanize_matplotlib

def gaussian_elimination_naive(A, b, verbose=True):
    A = np.array(A, dtype=float)
    b = np.array(b, dtype=float)
    n = len(b)
    Ab = np.hstack([A, b.reshape(-1, 1)])
    fmt = lambda x: f"{x:12.4e}" if ((abs(x) < 1e-3 and x != 0) or abs(x) >= 1e4) else f"{x:12.4f}"
    if verbose:
        print("【拡大係数行列の推移】")
        print(np.array2string(Ab, formatter={'float_kind': fmt}))
    for k in range(n - 1):
        if Ab[k, k] == 0.0:
            raise ZeroDivisionError(f"ピボット a_{{{k},{k}}} が 0 です。")
        for i in range(k + 1, n):
            m = Ab[i, k] / Ab[k, k]
            Ab[i, k:] -= m * Ab[k, k:]
            if verbose:
                print(f"Step {k+1}: 乗数 m = {m:.4e}")
                print(np.array2string(Ab, formatter={'float_kind': fmt}))
    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        x[i] = (Ab[i, -1] - np.dot(Ab[i, i+1:n], x[i+1:n])) / Ab[i, i]
    return x

# ウィルキンソン型の 2x2 連立一次方程式 (ε = 1e-16)
eps = 1e-16
A_wilkinson = [
    [eps, 1.0],
    [1.0, 1.0]
]
b_wilkinson = [1.0, 2.0]

x_true = np.array([1.0 / (1.0 - eps), (1.0 - 2.0 * eps) / (1.0 - eps)])

print(f"=== ピットフォール2: 微小ピボット (ε = {eps:.1e}) の実験 ===")
print(f"理論上の真の解: x_0 = {x_true[0]:.15f},  x_1 = {x_true[1]:.15f}")
print("-" * 65)

x_naive = gaussian_elimination_naive(A_wilkinson, b_wilkinson, verbose=True)
rel_error = np.linalg.norm(x_naive - x_true) / np.linalg.norm(x_true)

print(f"ナイーブ法による解: x_0 = {x_naive[0]:.15f},  x_1 = {x_naive[1]:.15f}")
print(f"相対誤差: {rel_error * 100:.2f}%")
print("-" * 65)
print("❌ 【情報落ちの恐怖】")
print(f"   真の解 x_0 = 1.0 に対して、計算された x_0 = {x_naive[0]:.4f}！")
print("   消去乗数 m = 1/ε = 10^16 という天文学的な数が、元の係数の情報を消し去りました。")


---
## 5. 解決方法：部分ピボット選択法（Partial Pivoting）

ゼロピボットと微小ピボットによる誤差爆発を一挙に解決する技法が **部分ピボット選択法（Partial Pivoting）** です。

### 部分ピボット選択のアルゴリズム
ステップ $k$ において前進消去を行う直前に、$k$ 列目の候補要素（$i = k, k+1, \dots, n-1$）の中から、**絶対値が最大の要素**を探します：

$$ p = \arg\max_{i = k, \dots, n-1} |a_{ik}^{(k)}| $$

もし $p \neq k$ であれば、**第 $k$ 行と第 $p$ 行を丸ごと交換**します（$R_k \longleftrightarrow R_p$）。

### なぜこれで数値的安定性が保証されるのか？
行を交換した後の新しいピボット $a_{kk}^{(k)}$ は列 $k$ の中で絶対値最大であるため、消去乗数は必ず：

$$ |m_{ik}| = \left| \frac{a_{ik}^{(k)}}{a_{kk}^{(k)}} \right| \le 1.0 \quad (\text{すべての } i > k \text{ に対して}) $$

**「乗数の絶対値が必ず $1$ 以下に抑えられる」** ことが数学的に保証されます！  
要素の肥大化が絶対に起きず、情報落ちや桁落ちによる誤差爆発を完全に封じ込めることができます。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import math
import japanize_matplotlib

def gaussian_elimination_pivot(A, b, verbose=True):
    """
    部分ピボット選択付きガウスの消去法 (Partial Pivoting)
    """
    A = np.array(A, dtype=float)
    b = np.array(b, dtype=float)
    n = len(b)
    
    if A.shape != (n, n):
        raise ValueError(f"係数行列 A の形状 {A.shape} と ベクトル b の次元 {n} が一致しません。")
    
    Ab = np.hstack([A, b.reshape(-1, 1)])
    fmt = lambda x: f"{x:12.4e}" if ((abs(x) < 1e-3 and x != 0) or abs(x) >= 1e4) else f"{x:12.4f}"
    
    if verbose:
        print("【初期の拡大係数行列 [A | b]】")
        print(np.array2string(Ab, formatter={'float_kind': fmt}))
        print("-" * 70)

    # 1. 前進消去 (Partial Pivoting)
    for k in range(n - 1):
        # 列 k の行 k 以降で絶対値最大の要素を持つ行を探す
        p = k + np.argmax(np.abs(Ab[k:, k]))
        pivot_val = abs(Ab[p, k])
        
        if pivot_val == 0.0:
            raise ValueError("ピボット最大値が 0 のため、行列は正則でなく解を一意に決定できません。")
        
        # 行の入れ替え
        if p != k:
            Ab[[k, p]] = Ab[[p, k]]
            if verbose:
                print(f"Step {k+1}: ピボット最大化のため 第 {k} 行 と 第 {p} 行 を交換 (選ばれたピボット: {Ab[k, k]:+.4f})")
        else:
            if verbose:
                print(f"Step {k+1}: 交換不要（現在の第 {k} 行が最大ピボット: {Ab[k, k]:+.4f}）")

        # 消去処理 (|m| <= 1.0 が常に保証される)
        for i in range(k + 1, n):
            m = Ab[i, k] / Ab[k, k]
            Ab[i, k:] -= m * Ab[k, k:]
            if verbose:
                m_str = f"{m:+.4e}" if (abs(m) < 1e-3 and m != 0) else f"{m:+.4f}"
                print(f"         第 {i} 行 <- 第 {i} 行 - ({m_str}) * 第 {k} 行  [|m| = {abs(m):.4f} <= 1.0]")
        
        if verbose:
            print(f"--- Step {k+1} 終了後の拡大係数行列 ---")
            print(np.array2string(Ab, formatter={'float_kind': fmt}))
            print("-" * 70)

    # 2. 後退代入
    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        if Ab[i, i] == 0.0:
            raise ZeroDivisionError(f"対角成分 a_{{{i},{i}}} が 0 のため解を一意に決定できません。")
        x[i] = (Ab[i, -1] - np.dot(Ab[i, i+1:n], x[i+1:n])) / Ab[i, i]

    return x

# 1. ゼロピボット問題への適用
A_zero = [[0.0, 2.0], [3.0, 4.0]]
b_zero = [4.0, 11.0]
print("=== 1. 落とし穴1（ゼロピボット問題）への適用 ===")
x_piv_zero = gaussian_elimination_pivot(A_zero, b_zero, verbose=True)
print(f"求まった解 x = {x_piv_zero}")
print("🎉 ゼロピボットを自動で回避し、正しい解 [1.0, 2.0] を得ることができました！\n")

# 2. 微小ピボット問題への適用
eps_val = 1e-16
A_wilk = [[eps_val, 1.0], [1.0, 1.0]]
b_wilk = [1.0, 2.0]
x_true = np.array([1.0 / (1.0 - eps_val), (1.0 - 2.0 * eps_val) / (1.0 - eps_val)])

print("=" * 70)
print(f"=== 2. 落とし穴2（微小ピボット問題 ε = {eps_val:.1e}）への適用 ===")
x_piv_wilk = gaussian_elimination_pivot(A_wilk, b_wilk, verbose=True)
err_piv = np.linalg.norm(x_piv_wilk - x_true) / np.linalg.norm(x_true)
print(f"求まった解 x = {x_piv_wilk}")
print(f"相対誤差: {err_piv:.4e}")
print("🎉 乗数が |m| = 1.0000e-16 <= 1.0 となったため情報落ちが全く起きず、倍精度の限界まで完全一致しました！")


---
### ガウスの消去法・ピボット選択のアルゴリズムをアニメーションで学ぶ

前進消去・行交換・後退代入の挙動をステップごとに直感的に確認できるHTMLアニメーション教材を用意しています：

- [https://kita-toshihiro.github.io/algoII/gauss1.html](https://kita-toshihiro.github.io/algoII/gauss1.html)


---
## 7. ガウスの消去での計算の実例

ガウスの消去法とピボット選択は、現代の産業界・工学システムの心臓部として機能しています。  
ここでは、データ分析（線形回帰）と、電子回路解析における具体例を紹介します。


---
### 7.1 重回帰分析

データサイエンスや機械学習において、目的変数 $\mathbf{y}$ を複数の説明変数行列 $X \in \mathbb{R}^{N \times d}$ で予測する **重回帰分析（Multiple Linear Regression）** を考えます：

$$ \mathbf{y} \approx X \mathbf{w} $$

最小二乗法（二乗誤差 $\|\mathbf{y} - X\mathbf{w}\|^2$ を最小化する）による最適パラメータ $\mathbf{w}$ は、微分の停留条件から導かれる **正規方程式（Normal Equation）** の解として与えられます：

$$ (X^T X) \mathbf{w} = X^T \mathbf{y} $$

これは、係数行列 $A = X^T X \in \mathbb{R}^{d \times d}$、右辺ベクトル $\mathbf{b} = X^T \mathbf{y} \in \mathbb{R}^d$ と置けば、まさに **連立一次方程式 $A \mathbf{w} = \mathbf{b}$ そのもの** です。  



#### ⚠️ 分析の際の落とし穴：多重共線性（Multicollinearity）
データセットの中に「ほぼ同じ情報を持つ2つの特徴量」（例：摂氏温度と華氏温度、身長と座高など、相関係数がほぼ $1.0$ の変数）が存在すると、$X^T X$ の列同士がほぼ一次従属となり、**行列の条件数が極端に悪化（最小ピボットが $0$ に接近）** します。  
ピボット選択を行わないと、回帰係数の推定値が $+10^8$ と $-10^8$ のように天文学的に爆発してしまいます。ピボット選択付きガウス消去法がどのように数値を安定させるか、実験してみましょう。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import math
import japanize_matplotlib

def gaussian_elimination_naive(A, b):
    A = np.array(A, dtype=float)
    b = np.array(b, dtype=float)
    n = len(b)
    Ab = np.hstack([A, b.reshape(-1, 1)])
    for k in range(n - 1):
        if Ab[k, k] == 0.0:
            raise ZeroDivisionError()
        for i in range(k + 1, n):
            m = Ab[i, k] / Ab[k, k]
            Ab[i, k:] -= m * Ab[k, k:]
    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        x[i] = (Ab[i, -1] - np.dot(Ab[i, i+1:n], x[i+1:n])) / Ab[i, i]
    return x

def gaussian_elimination_pivot(A, b):
    A = np.array(A, dtype=float)
    b = np.array(b, dtype=float)
    n = len(b)
    Ab = np.hstack([A, b.reshape(-1, 1)])
    for k in range(n - 1):
        p = k + np.argmax(np.abs(Ab[k:, k]))
        if abs(Ab[p, k]) == 0.0:
            raise ValueError("特異行列です。")
        if p != k:
            Ab[[k, p]] = Ab[[p, k]]
        for i in range(k + 1, n):
            m = Ab[i, k] / Ab[k, k]
            Ab[i, k:] -= m * Ab[k, k:]
    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        x[i] = (Ab[i, -1] - np.dot(Ab[i, i+1:n], x[i+1:n])) / Ab[i, i]
    return x

# 多重共線性（強い相関）を持つデータセットを生成
np.random.seed(42)
N = 100
# 特徴量 x1
x1 = np.random.randn(N)
# 特徴量 x2: x1 と強い相関 (微小ノイズ 1e-7)
x2 = x1 + 1e-7 * np.random.randn(N)

# 説明変数行列 X = [1, x1, x2]
X = np.column_stack([np.ones(N), x1, x2])
# 真の回帰係数: 切片 1.5, x1 の重み 2.0, x2 の重み 3.0
w_true = np.array([1.5, 2.0, 3.0])
y = X @ w_true + 1e-4 * np.random.randn(N)

# 正規方程式の係数行列 A = X^T X, 右辺 b = X^T y
A_norm = X.T @ X
b_norm = X.T @ y

cond_num = np.linalg.cond(A_norm)
print("=== 重回帰分析の正規方程式 ===")
print(f"真の重みパラメータ w: 切片={w_true[0]}, w1={w_true[1]}, w2={w_true[2]} (w1 + w2 = 5.0)")
print(f"正規方程式の係数行列 A = X^T X の条件数: {cond_num:.2e} (極めて悪条件)")
print("-" * 65)

# 部分ピボット選択付き消去法で解く
w_pivot = gaussian_elimination_pivot(A_norm, b_norm)
# NumPy標準
try:
    w_numpy = np.linalg.solve(A_norm, b_norm)
    diff_np = f"{np.linalg.norm(w_pivot - w_numpy):.4e}"
except np.linalg.LinAlgError:
    diff_np = "NumPy 特異行列例外"

print(f"🎯 部分ピボット選択法による回帰係数:")
print(f"   切片 = {w_pivot[0]:.4f},  w1 = {w_pivot[1]:.4f},  w2 = {w_pivot[2]:.4f}")
print(f"   (w1 + w2 の合計 = {w_pivot[1] + w_pivot[2]:.4f} ≈ 5.0 となり、統合効果を正しく捉えている)")
print(f"   NumPy solve との差: {diff_np}")
print("-" * 65)


--- 

### 7.1 節点電位法（Node Voltage Analysis）による回路網解析
電子回路CADや回路シミュレータ（SPICEなど）では、回路の各接続点（節点）の電位を求めるために **キルヒホッフの電流則（KCL）** を適用します。

各節点 $i$ に流れ込む電流の総和はゼロであるため、コンダクタンス（抵抗の逆数 $g = 1/R$）を行列 $G$ としてまとめると、次の**節点方程式**が得られます：

$$ G \mathbf{v} = \mathbf{i} $$

- $G \in \mathbb{R}^{n \times n}$: 節点コンダクタンス行列
- $\mathbf{v} \in \mathbb{R}^n$: 求める各節点の電位ベクトル $[v_1, v_2, \dots, v_n]^T$
- $\mathbf{i} \in \mathbb{R}^n$: 節点に接続された外部電流源ベクトル

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import math
import japanize_matplotlib

def gaussian_elimination_pivot(A, b):
    A = np.array(A, dtype=float)
    b = np.array(b, dtype=float)
    n = len(b)
    Ab = np.hstack([A, b.reshape(-1, 1)])
    for k in range(n - 1):
        p = k + np.argmax(np.abs(Ab[k:, k]))
        if abs(Ab[p, k]) == 0.0:
            raise ValueError("特異行列です。")
        if p != k:
            Ab[[k, p]] = Ab[[p, k]]
        for i in range(k + 1, n):
            m = Ab[i, k] / Ab[k, k]
            Ab[i, k:] -= m * Ab[k, k:]
    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        x[i] = (Ab[i, -1] - np.dot(Ab[i, i+1:n], x[i+1:n])) / Ab[i, i]
    return x

# 【3節点 抵抗回路網の節点電位法】
# 節点1: 電流源 I_s1 = 2.0 A が流入, R1 = 10 Ω (接地間), R2 = 20 Ω (節点1-2間)
# 節点2: R3 = 30 Ω (接地間), R4 = 40 Ω (節点2-3間)
# 節点3: 電流源 I_s2 = 1.0 A が流出 (-1.0 A), R5 = 50 Ω (接地間)

# コンダクタンス g = 1 / R
g1, g2, g3, g4, g5 = 1/10.0, 1/20.0, 1/30.0, 1/40.0, 1/50.0

# 節点方程式 G * v = i
G = np.array([
    [g1 + g2,    -g2,          0.0],
    [   -g2,  g2 + g3 + g4,   -g4 ],
    [   0.0,     -g4,       g4 + g5]
])
i_src = np.array([2.0, 0.0, -1.0])

print("コンダクタンス行列 G [S]:")
print(G)
print(f"流入電流ベクトル i [A]: {i_src}")
print("-" * 65)

# ピボット選択付きガウス消去法で節点電圧を解く
v_nodes = gaussian_elimination_pivot(G, i_src)

print(f"🎯 計算された各節点の電位:")
print(f"   節点1の電位 v_1 = {v_nodes[0]:8.4f} V")
print(f"   節点2の電位 v_2 = {v_nodes[1]:8.4f} V")
print(f"   節点3の電位 v_3 = {v_nodes[2]:8.4f} V")

# 検算: G * v - i
#res_circuit = np.dot(G, v_nodes) - i_src
#print(f"キルヒホッフ電流則の残差 (KCL誤差): {np.linalg.norm(res_circuit):.4e} A")


---
## （参考）LU分解

### 実務の標準形式：LU分解（$PA = LU$）
実務のシミュレーション（構造解析、熱伝導、物理シミュレーションなど）では、**「行列 $A$（系の構造）は一定のまま、右辺 $\mathbf{b}$（外力や入力条件）だけを何万回も変えて解く」** という状況が頻出します。  
もしガウスの消去法を愚直に繰り返すと、$\mathbf{b}$ が変わるたびに毎回 $O(n^3)$ の重い前進消去をやり直さなければなりません。

そこで実務では **LU分解** を利用します：
- 前進消去で用いた乗数 $m_{ik}$ を下三角行列 $L$（Unit Lower Triangular）に格納し、消去結果の上三角行列を $U$（Upper Triangular）とする。
- ピボット選択の行交換を行列 $P$（Permutation Matrix）として表現する：
  $$ PA = LU $$
- $PA = LU$ の分解には $O(n^3)$ かかりますが、**一度分解してしまえば、任意の新しい $\mathbf{b}$ に対しては前進代入と後退代入のみでわずか $O(n^2)$ で超高速に解けます**。

--- 
--- 

（この資料は、OpenCode + Gemini 3.8 Flash を用いて生成しリライトしたものです 喜多　敏博）